# Model Runs: E-OBS & ECMWF Predictive Modeling

**Purpose**: Train and evaluate multiple models to predict:
- `eobs_wet_day_frequency` (WDF)
- `eobs_spi_1_values` (SPI-1)

**Approach**:
- Leave-One-Out Cross-Validation (LOOCV)
- Multiple models: Linear, Regularized, Tree-based, Neural Networks
- Multiple input configurations: ECMWF, Lagged E-OBS, Combined
- Both pooled and seasonal models


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os
import time
import json
from tqdm import tqdm

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

✓ Libraries imported


# 1. Load Data


In [ ]:
# Load merged data (single location, temporal only)
data_path = "000000 Final Data/eastnor/3_eobs_ecmwf_merged_with_seasons.csv"
df = pd.read_csv(data_path)

df['time'] = pd.to_datetime(df['time'])
display(df.head())

✓ Data loaded successfully
  Shape: (368, 19)
  Time range: 1993-03-31 00:00:00 to 2023-12-31 00:00:00
  Seasons: ['Spring' 'Summer' 'Fall' 'Winter']

Columns: ['time', 'season', 'eobs_press', 'eobs_press_-1', 'eobs_press_-2', 'eobs_precip', 'eobs_precip_-1', 'eobs_precip_-2', 'eobs_temp', 'eobs_temp_-1', 'eobs_temp_-2', 'eobs_wet_day_frequency', 'eobs_spi_1_values', 'ecmwf_temp_ensmean', 'ecmwf_precip_ensmean', 'ecmwf_press_ensmean', 'ecmwf_temp_ensvar', 'ecmwf_precip_ensvar', 'ecmwf_press_ensvar']


,time,season,eobs_press,eobs_press_-1,eobs_press_-2,eobs_precip,eobs_precip_-1,eobs_precip_-2,eobs_temp,eobs_temp_-1,eobs_temp_-2,eobs_wet_day_frequency,eobs_spi_1_values,ecmwf_temp_ensmean,ecmwf_precip_ensmean,ecmwf_press_ensmean,ecmwf_temp_ensvar,ecmwf_precip_ensvar,ecmwf_press_ensvar
0,1993-03-31,Spring,1014.49760,1018.13860,998.94730,6.095313,25.704689,38.970314,-3.907354,-4.456501,-5.261134,0.121976,-1.862051,-6.818393,45.683977,1009.978842,1.784359,232.807868,34.732208
1,1993-04-30,Spring,1014.26917,1014.49760,1018.13860,9.868751,6.095313,25.704689,0.896708,-3.907354,-4.456501,0.151042,-1.329268,-2.247027,59.951233,1008.791816,0.943828,254.561116,9.658475
2,1993-05-31,Spring,1015.96890,1014.26917,1014.49760,82.575005,9.868751,6.095313,7.252682,0.896708,-3.907354,0.433972,1.124866,5.869891,74.586936,1017.762445,1.762311,1214.230474,4.808951
3,1993-06-30,Summer,1010.77783,1015.96890,1014.26917,29.423439,82.575005,9.868751,7.514687,7.252682,0.896708,0.318750,-1.447206,9.482227,88.519095,1013.450769,1.737408,1034.173241,3.752825
4,1993-07-31,Summer,1005.39110,1010.77783,1015.96890,142.346880,29.423439,82.575005,10.455398,7.514687,7.252682,0.658266,1.380052,10.713787,116.226984,1008.695492,1.257259,587.099239,5.320805


# 2. Model Library


In [ ]:
models = {
    # ═══════════════════════════════════════════════════════════════
    # LINEAR MODELS
    # ═══════════════════════════════════════════════════════════════
    "LinearRegression": lambda: make_pipeline(StandardScaler(), LinearRegression()),
    
    # Ridge Regression (L2 regularization)
    "Ridge_alpha_0.1": lambda: make_pipeline(StandardScaler(), Ridge(alpha=0.1)),
    "Ridge_alpha_1.0": lambda: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Ridge_alpha_10.0": lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "Ridge_alpha_100.0": lambda: make_pipeline(StandardScaler(), Ridge(alpha=100.0)),
    
    # Lasso Regression (L1 regularization)
    "Lasso_alpha_0.0001": lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.0001, max_iter=10000)),
    "Lasso_alpha_0.001": lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.001, max_iter=10000)),
    "Lasso_alpha_0.01": lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=10000)),
    "Lasso_alpha_0.1": lambda: make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000)),
    "Lasso_alpha_1.0": lambda: make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=10000)),
    
    # Elastic Net (L1 + L2)
    "ElasticNet": lambda: make_pipeline(StandardScaler(), ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000)),
    
    # Bayesian Ridge
    "BayesianRidge": lambda: make_pipeline(StandardScaler(), BayesianRidge()),
    
    # ═══════════════════════════════════════════════════════════════
    # RANDOM FOREST (various depths and leaf sizes)
    # ═══════════════════════════════════════════════════════════════
    "RF_depth3_leaf5": lambda: RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=5, random_state=42),
    "RF_depth4_leaf5": lambda: RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=5, random_state=42),
    "RF_depth4_leaf10": lambda: RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=10, random_state=42),
    "RF_depth5_leaf10": lambda: RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_leaf=10, random_state=42),
    
    # Gradient Boosting
    "GradientBoosting": lambda: GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42),
    
    # ═══════════════════════════════════════════════════════════════
    # NEURAL NETWORKS (MLP)
    # ═══════════════════════════════════════════════════════════════
    # Tiny (2 neurons)
    "MLP_tiny": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(2,), activation="relu", alpha=1e-1, learning_rate_init=1e-3, early_stopping=True, n_iter_no_change=15, max_iter=3000, random_state=42)),
    "MLP_tiny_tanh": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(2,), activation="tanh", alpha=1e-1, learning_rate_init=1e-3, early_stopping=True, n_iter_no_change=15, max_iter=3000, random_state=42)),
    
    # Small (4 neurons)
    "MLP_small": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(4,), activation="relu", alpha=1e-1, learning_rate_init=3e-3, early_stopping=True, n_iter_no_change=20, max_iter=4000, random_state=42)),
    "MLP_small_tanh": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(4,), activation="tanh", alpha=1e-1, learning_rate_init=3e-3, early_stopping=True, n_iter_no_change=20, max_iter=4000, random_state=42)),
    
    # Medium (6 neurons)
    "MLP_medium": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(6,), activation="relu", alpha=5e-2, learning_rate_init=3e-3, early_stopping=True, n_iter_no_change=20, max_iter=5000, random_state=42)),
    "MLP_medium_tanh": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(6,), activation="tanh", alpha=5e-2, learning_rate_init=3e-3, early_stopping=True, n_iter_no_change=20, max_iter=5000, random_state=42)),
    
    # Two layers
    "MLP_2layer_small": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(4,2), activation="relu", alpha=1e-1, learning_rate_init=3e-3, early_stopping=True, n_iter_no_change=20, max_iter=4000, random_state=42)),
    "MLP_2layer_small_tanh": lambda: make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(4,2), activation="tanh", alpha=1e-1, learning_rate_init=3e-3, early_stopping=True, n_iter_no_change=20, max_iter=4000, random_state=42)),
}
linear_models = [k for k in models.keys() if any(x in k for x in ['Linear', 'Ridge', 'Lasso', 'Elastic', 'Bayesian'])]
rf_models = [k for k in models.keys() if 'RF' in k or 'Gradient' in k]
mlp_models = [k for k in models.keys() if 'MLP' in k]

✓ Model library created: 25 models

Model types:
  Linear/Regularized: 12
  Random Forest/Boosting: 5
  Neural Networks: 8


# 3. Input Feature Configurations

**Note**: The following configurations are defined based on:
1. Standard baseline comparisons (E-OBS vs ECMWF vs Combined)
2. Somewhat **data-driven combinations** from correlation analysis in `4_merged_analysis.ipynb`

We test:
- Variable-specific configs (precip-only)
- Alternative lag horizons (lag-2)
- Strategic combinations (variance + lags, precip-focused)

In [ ]:
# Define input feature sets
input_configs = {
    # ═══════════════════════════════════════════════════════════════
    # E-OBS ONLY (baseline - what if we had current observations?)
    # ═══════════════════════════════════════════════════════════════
    "eobs": ["eobs_press", "eobs_temp", "eobs_precip"],
    
    # ═══════════════════════════════════════════════════════════════
    # E-OBS LAGGED (persistence baseline)
    # ═══════════════════════════════════════════════════════════════
    "eobs_lag1": ["eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1"],
    "eobs_lag2": ["eobs_press_-2", "eobs_temp_-2", "eobs_precip_-2"],
    "eobs_lag1_lag2": [
        "eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1",
        "eobs_press_-2", "eobs_temp_-2", "eobs_precip_-2"
    ],
    
    # ═══════════════════════════════════════════════════════════════
    # ECMWF FORECASTS
    # ═══════════════════════════════════════════════════════════════
    "ecmwf_mean": ["ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean"],
    "ecmwf_var": ["ecmwf_press_ensvar", "ecmwf_temp_ensvar", "ecmwf_precip_ensvar"],
    "ecmwf_mean_var": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "ecmwf_press_ensvar", "ecmwf_temp_ensvar", "ecmwf_precip_ensvar"
    ],
    "ecmwf_mean_precipvar": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "ecmwf_precip_ensvar"
    ],
    
    # ═══════════════════════════════════════════════════════════════
    # COMBINED: ECMWF + LAGGED E-OBS
    # ═══════════════════════════════════════════════════════════════
    "ecmwf_mean_eobs_lag1": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1"
    ],
    "ecmwf_mean_var_eobs_lag1": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "ecmwf_press_ensvar", "ecmwf_temp_ensvar", "ecmwf_precip_ensvar",
        "eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1"
    ],
    
    # ═══════════════════════════════════════════════════════════════
    # DATA-DRIVEN COMBINATIONS (Based on 4_merged_analysis.ipynb)
    # ═══════════════════════════════════════════════════════════════
    # These combinations are motivated by correlation analysis findings:
    # - Precipitation dominates both responses (strongest predictor)
    # - Lag-2 may provide complementary information to lag-1
    # - ECMWF uncertainty shows potential predictive power
    # - Strategic mix of forecast + persistence for complementarity
    
    # Variable-specific (precip-only)
    "ecmwf_precip_only": [
        "ecmwf_precip_ensmean", "ecmwf_precip_ensvar"
    ],
    "eobs_precip_lags_only": [
        "eobs_precip_-1", "eobs_precip_-2"
    ],
    
    # Alternative lag horizon
    "ecmwf_mean_eobs_lag2": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "eobs_press_-2", "eobs_temp_-2", "eobs_precip_-2"
    ],
    
    # Both lag horizons - tests if lag-1 and lag-2 are complementary
    "ecmwf_mean_both_lags": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1",
        "eobs_press_-2", "eobs_temp_-2", "eobs_precip_-2"
    ],
    
    # Focused on precipitation (forecast + lags)
    "ecmwf_precip_eobs_precip_lags": [
        "ecmwf_precip_ensmean", "ecmwf_precip_ensvar",
        "eobs_precip_-1", "eobs_precip_-2"
    ],
    
    # Variance + lags
    "ecmwf_var_eobs_lag1": [
        "ecmwf_press_ensvar", "ecmwf_temp_ensvar", "ecmwf_precip_ensvar",
        "eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1"
    ],
    
    # Everything (let models do feature selection)
    "all_features": [
        "ecmwf_press_ensmean", "ecmwf_temp_ensmean", "ecmwf_precip_ensmean",
        "ecmwf_press_ensvar", "ecmwf_temp_ensvar", "ecmwf_precip_ensvar",
        "eobs_press_-1", "eobs_temp_-1", "eobs_precip_-1",
        "eobs_press_-2", "eobs_temp_-2", "eobs_precip_-2"
    ],
}

output_features = {
    "wdf": "eobs_wet_day_frequency",
    "spi-1": "eobs_spi_1_values",
}
season_options = ["Winter", "Spring", "Summer", "Fall", None]



✓ Configuration library created
  Input configurations: 17

  Standard configurations: 10
    eobs: 3 features
    eobs_lag1: 3 features
    eobs_lag2: 3 features
    eobs_lag1_lag2: 6 features
    ecmwf_mean: 3 features
    ecmwf_var: 3 features
    ecmwf_mean_var: 6 features
    ecmwf_mean_precipvar: 4 features
    ecmwf_mean_eobs_lag1: 6 features
    ecmwf_mean_var_eobs_lag1: 9 features

  Data-driven configurations: 7
    ecmwf_precip_only: 2 features
    eobs_precip_lags_only: 2 features
    ecmwf_mean_eobs_lag2: 6 features
    ecmwf_mean_both_lags: 9 features
    ecmwf_precip_eobs_precip_lags: 4 features
    ecmwf_var_eobs_lag1: 6 features
    all_features: 12 features

  Output options: ['wdf', 'spi-1']
  Season options: ['Winter', 'Spring', 'Summer', 'Fall', None]

  Max feature count: 12 (all_features)


# 4. Core Functions

In [ ]:
def run_loocv(df_subset, model_factory, input_features, output_col):
    """
    Run Leave-One-Out Cross-Validation.
    
    Parameters:
    df_subset : DataFrame
        Data to use (can be filtered by season)
    model_factory : callable
        Function that returns unfitted model
    input_features : list[str]
        Column names for X
    output_col : str
        Column name for y
    
    Returns:
    DataFrame with columns: time, y_pred, y_true
    """
    X = df_subset[input_features].values
    y = df_subset[output_col].values
    times = df_subset['time'].values
    
    n = len(X)
    y_preds = np.zeros(n)
    
    for i in range(n):
        # Leave one out
        X_train = np.delete(X, i, axis=0)
        y_train = np.delete(y, i)
        X_test = X[i:i+1]
        
        # Fit and predict
        model = model_factory()
        model.fit(X_train, y_train)
        y_preds[i] = model.predict(X_test)[0]
    
    return pd.DataFrame({
        'time': times,
        'y_pred': y_preds,
        'y_true': y
    })

def calculate_baseline(df_subset, output_col):
    """
    Calculate baseline LOOCV RMSE (predicting mean of training data).
    
    Returns:
    float: baseline RMSE
    """
    y = df_subset[output_col].values
    n = len(y)
    squared_errors = []
    
    for i in range(n):
        y_train = np.delete(y, i)
        y_test = y[i]
        y_pred = np.mean(y_train)
        squared_errors.append((y_test - y_pred)**2)
    
    return np.sqrt(np.mean(squared_errors))

def generate_filename(model_name, input_name, output_name, season):
    """
    Generate automatic filename from configuration (FENNO approach).
    
    Example: run_20241228_143052--Ridge_alpha_1.0--ecmwf_mean--wdf--all.csv
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    season_str = season.lower() if season else "all"
    
    # Clean names for filesystem
    model_clean = model_name.replace(".", "_")
    input_clean = input_name.replace(".", "_")
    output_clean = output_name
    
    return f"run_{timestamp}--{model_clean}--{input_clean}--{output_clean}--{season_str}.csv"

def get_config_signature(model_name, input_name, output_name, season):
    """
    Get configuration signature (filename without timestamp).
    
    Used to check if this configuration was already run.
    
    Example: Ridge_alpha_1_0--ecmwf_mean--wdf--all
    """
    season_str = season.lower() if season else "all"
    model_clean = model_name.replace(".", "_")
    input_clean = input_name.replace(".", "_")
    
    return f"{model_clean}--{input_clean}--{output_name}--{season_str}"

def check_if_already_run(model_name, input_name, output_name, season, results_dir):
    """
    Check if this configuration was already run (ignoring timestamp).
    
    Returns:
    tuple: (bool, str or None)
        - True/False: Whether config exists
        - Existing filename if found, None otherwise
    """
    config_sig = get_config_signature(model_name, input_name, output_name, season)
    
    predictions_dir = f"{results_dir}/predictions"
    if not os.path.exists(predictions_dir):
        return False, None
    
    # Check all files in predictions directory
    for filename in os.listdir(predictions_dir):
        if filename.startswith('run_') and filename.endswith('.csv'):
            # Extract config part (everything after timestamp)
            parts = filename.split('--', 1)  # Split on first '--'
            if len(parts) == 2:
                # Remove 'run_TIMESTAMP' prefix and '.csv' suffix
                existing_config = parts[1].replace('.csv', '')
                
                if existing_config == config_sig:
                    return True, filename
    
    return False, None

✓ Core functions defined
  - run_loocv()
  - calculate_baseline()
  - generate_filename()
  - get_config_signature()
  - check_if_already_run()


### NON-SEASONAL

In [99]:
# # Define all experiments to run
# # Each experiment is one model + input + output + season combination

# # ═══════════════════════════════════════════════════════════════════════════════
# # Motivated by correlation analysis (see 4_merged_analysis.ipynb Section 8):
# # - Precipitation dominates both responses (r≈0.5 for WDF, r≈0.72 for SPI-1)
# # - Lag-2 shows different complementarity patterns than lag-1
# # - ECMWF variance potentially informative (uncertainty as predictor)
# # - Strategic variable-specific combinations reduce noise from weak predictors
# # ═══════════════════════════════════════════════════════════════════════════════

# experiments = []

# # ═══════════════════════════════════════════════════════════════════════════════
# # GROUP 1: BASELINE LINEAR MODELS (Pooled - all seasons)
# # ═══════════════════════════════════════════════════════════════════════════════
# # Test LinearRegression with ALL input configurations for fair comparison
# print("Defining experiments...")

# all_input_configs = list(input_configs.keys())

# for output in ['wdf', 'spi-1']:
#     # Test EVERY input configuration with basic linear regression
#     for input_name in all_input_configs:
#         experiments.append({
#             'model': 'LinearRegression',
#             'input': input_name,
#             'output': output,
#             'season': None,
#             'group': '1_baseline_linear_pooled'
#         })

# # ═══════════════════════════════════════════════════════════════════════════════
# # GROUP 2: REGULARIZATION COMPARISON (Pooled)
# # ═══════════════════════════════════════════════════════════════════════════════
# # Test ALL regularization models (Ridge, Lasso, ElasticNet, BayesianRidge) with ALL input configurations
# regularization_models = [
#     'Ridge_alpha_0.1', 'Ridge_alpha_1.0', 'Ridge_alpha_10.0', 'Ridge_alpha_100.0',
#     'Lasso_alpha_0.0001', 'Lasso_alpha_0.001', 'Lasso_alpha_0.01', 'Lasso_alpha_0.1', 'Lasso_alpha_1.0',
#     'ElasticNet',
#     'BayesianRidge'
# ]

# for output in ['wdf', 'spi-1']:
#     for input_name in all_input_configs:
#         for model in regularization_models:
#             experiments.append({
#                 'model': model,
#                 'input': input_name,
#                 'output': output,
#                 'season': None,
#                 'group': '2_regularization_pooled'
#             })

# # ═══════════════════════════════════════════════════════════════════════════════
# # GROUP 3: NON-LINEAR MODELS (Pooled)
# # ═══════════════════════════════════════════════════════════════════════════════
# # Test ALL Random Forest and Gradient Boosting models with selected input configurations
# nonlinear_models = [
#     'RF_depth3_leaf5', 'RF_depth4_leaf5', 'RF_depth4_leaf10', 'RF_depth5_leaf10',
#     'GradientBoosting'
# ]

# nonlinear_input_configs = [
#     'ecmwf_mean',
#     'ecmwf_mean_eobs_lag1',
#     'ecmwf_mean_var',
#     'ecmwf_mean_var_eobs_lag1',
#     'ecmwf_mean_both_lags',
#     'all_features',
#     'ecmwf_precip_eobs_precip_lags',
#     'eobs_lag1_lag2',
#     'ecmwf_var_eobs_lag1',
#     'ecmwf_precip_only',
#     'eobs_precip_lags_only'
# ]

# for output in ['wdf', 'spi-1']:
#     for input_name in nonlinear_input_configs:
#         for model in nonlinear_models:
#             experiments.append({
#                 'model': model,
#                 'input': input_name,
#                 'output': output,
#                 'season': None,
#                 'group': '3_nonlinear_pooled'
#             })

# # ═══════════════════════════════════════════════════════════════════════════════
# # GROUP 4: NEURAL NETWORKS (Pooled)
# # ═══════════════════════════════════════════════════════════════════════════════
# # Test ALL MLP models (various sizes, activations, depths) with selected input configurations
# mlp_models = [
#     'MLP_tiny', 'MLP_tiny_tanh',
#     'MLP_small', 'MLP_small_tanh',
#     'MLP_medium', 'MLP_medium_tanh',
#     'MLP_2layer_small', 'MLP_2layer_small_tanh'
# ]

# for output in ['wdf', 'spi-1']:
#     for input_name in nonlinear_input_configs:
#         for model in mlp_models:
#             experiments.append({
#                 'model': model,
#                 'input': input_name,
#                 'output': output,
#                 'season': None,
#                 'group': '4_mlp_pooled'
#             })

# # ═══════════════════════════════════════════════════════════════════════════════
# # GROUP 5: ENSEMBLE VARIANCE TESTS (Does uncertainty help?)
# # ═══════════════════════════════════════════════════════════════════════════════
# for output in ['wdf', 'spi-1']:
#     for input_name in ['ecmwf_var', 'ecmwf_mean_var', 'ecmwf_mean_precipvar']:
#         experiments.append({
#             'model': 'LinearRegression',
#             'input': input_name,
#             'output': output,
#             'season': None,
#             'group': '5_ensemble_variance'
#         })

In [100]:
# print(f"\n✓ Experiments defined: {len(experiments)} total")
# print(f"\nBreakdown by group:")
# for group in ['1_baseline_linear_pooled', '2_regularization_pooled', '3_nonlinear_pooled', 
#               '4_mlp_pooled', '5_ensemble_variance', '6_data_driven_combinations', '7_seasonal_models']:
#     count = sum(1 for exp in experiments if exp['group'] == group)
#     print(f"  {group:30s}: {count:3d} experiments")

# print(f"\n⚠ NOTE: Group 7 (seasonal models) is currently commented out.")
# print(f"   Workflow: Run Groups 1-6 → Review → Define best models → Run Group 7")

### SEASONAL

In [ ]:
experiments = []
all_input_configs = list(input_configs.keys())

# Define seasonal models (10 models × 5 inputs × 4 seasons × 2 responses = 400 experiments)
seasonal_models = [
    # Linear baseline 
    'LinearRegression',
    
    # Ridge, L2 regularization
    'Ridge_alpha_0.1',      
    'Ridge_alpha_1.0',      
    'Ridge_alpha_10.0',     
    
    # Lasso, L1 regularization
    'Lasso_alpha_0.0001',   
    'Lasso_alpha_0.001',    
    'Lasso_alpha_0.01',     
    
    # MLP, Non-linear models
    'MLP_small_tanh',       
    'MLP_medium_tanh',     
    'MLP_2layer_small_tanh', 
]

seasonal_inputs = [
    'eobs',
    'eobs_lag1',              
    'ecmwf_mean',             
    'ecmwf_mean_eobs_lag1',   
    'ecmwf_mean_eobs_lag2',   
    'ecmwf_mean_var'          
]

for output in ['wdf', 'spi-1']:
    for input_name in seasonal_inputs:
        for model in seasonal_models:
            for season in ['Winter', 'Spring', 'Summer', 'Fall']:  # Match data column values
                experiments.append({
                    'model': model,
                    'input': input_name,
                    'output': output,
                    'season': season,
                    'group': '7_seasonal_models'
                })

Defining experiments...
✓ Added GROUP 7: SEASONAL MODELS
  Models: 10 (LinearRegression, Ridge×3, Lasso×3, MLP×3)
  Inputs: 6 (persistence, forecasts, blends, uncertainty)
  Total seasonal experiments: 480 (10 models × 5 inputs × 4 seasons × 2 responses)

✓ Experiments defined: 480 total

Breakdown by group:
  1_baseline_linear_pooled      :   0 experiments
  2_regularization_pooled       :   0 experiments
  3_nonlinear_pooled            :   0 experiments
  4_mlp_pooled                  :   0 experiments
  5_ensemble_variance           :   0 experiments
  6_data_driven_combinations    :   0 experiments
  7_seasonal_models             : 480 experiments


# 6. Compute Baselines (Run Once)

In [ ]:
baseline_dir = "000000 Final Data/eastnor/model_results/baselines"
os.makedirs(baseline_dir, exist_ok=True)

baselines = {}
for output_key, output_col in output_features.items():
    for season in season_options:
        # Filter data if seasonal
        if season:
            df_subset = df[df['season'] == season].copy()
        else:
            df_subset = df.copy()
        
        # Calculate baseline
        baseline_rmse = calculate_baseline(df_subset, output_col)
        
        # Save
        season_str = season.lower() if season else "all"
        baseline_key = f"{output_key}--{season_str}"
        baselines[baseline_key] = baseline_rmse
        
        print(f"  {baseline_key:20s}: RMSE={baseline_rmse:.6f} (n={len(df_subset)})")

# Save baselines to JSON
baseline_file = f"{baseline_dir}/baselines_all.json"
with open(baseline_file, 'w') as f:
    json.dump(baselines, f, indent=2)


COMPUTING BASELINES
  wdf--winter         : RMSE=0.117985 (n=91)
  wdf--spring         : RMSE=0.130723 (n=93)
  wdf--summer         : RMSE=0.139524 (n=93)
  wdf--fall           : RMSE=0.118894 (n=91)
  wdf--all            : RMSE=0.137510 (n=368)
  spi-1--winter       : RMSE=0.910427 (n=91)
  spi-1--spring       : RMSE=1.026811 (n=93)
  spi-1--summer       : RMSE=0.943263 (n=93)
  spi-1--fall         : RMSE=1.079320 (n=91)
  spi-1--all          : RMSE=0.988240 (n=368)

✓ Baselines computed and saved to: 000000 Final Data/eastnor/model_results/baselines/baselines_all.json


# 7. RUN ALL EXPERIMENTS

In [ ]:
from IPython.display import clear_output

results_dir = "000000 Final Data/eastnor/model_results"
os.makedirs(f"{results_dir}/predictions", exist_ok=True)
os.makedirs(f"{results_dir}/metadata", exist_ok=True)

runlog_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
runlog_path = f"{results_dir}/{runlog_timestamp}_runlog.txt"
runlog = open(runlog_path, 'w')

all_results = []
failed_experiments = []
skipped_experiments = []

for i, exp in enumerate(tqdm(experiments, desc="Progress")):
    # Clear output every 15 experiments to keep notebook small
    if i > 0 and i % 15 == 0:
        clear_output(wait=True)
    
    # Check if this configuration was already run (skip if exists)
    already_run, existing_file = check_if_already_run(
        exp['model'], exp['input'], exp['output'], exp['season'], results_dir
    )
    
    if already_run:
        exp_desc = f"{exp['model']}--{exp['input']}--{exp['output']}--{exp['season'] or 'all'}"
        skipped_experiments.append({**exp, 'existing_file': existing_file})
        continue
    
    try:
        # Print current experiment
        exp_desc = f"{exp['model']}--{exp['input']}--{exp['output']}--{exp['season'] or 'all'}"
        
        if exp['season']:
            df_subset = df[df['season'] == exp['season']].copy()
        else:
            df_subset = df.copy()
        
        # Run LOOCV
        start_time = time.time()
        
        result_df = run_loocv(
            df_subset=df_subset,
            model_factory=models[exp['model']],
            input_features=input_configs[exp['input']],
            output_col=output_features[exp['output']]
        )
        
        elapsed = time.time() - start_time
        
        rmse = np.sqrt(((result_df['y_pred'] - result_df['y_true'])**2).mean())
        mae = (result_df['y_pred'] - result_df['y_true']).abs().mean()
        
        # Calculate R²
        ss_res = ((result_df['y_true'] - result_df['y_pred'])**2).sum()
        ss_tot = ((result_df['y_true'] - result_df['y_true'].mean())**2).sum()
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # Get baseline for skill score
        season_str = exp['season'].lower() if exp['season'] else "all"
        baseline_key = f"{exp['output']}--{season_str}"
        baseline_rmse = baselines.get(baseline_key, np.nan)
        skill_score = 1 - (rmse / baseline_rmse) if baseline_rmse > 0 else 0
        
        # Generate filename
        filename = generate_filename(
            exp['model'], exp['input'], exp['output'], exp['season']
        )
        
        # Save predictions (CSV)
        result_df.to_csv(f"{results_dir}/predictions/{filename}", index=False)
        
        # Save metadata (JSON)
        metadata = {
            'model': exp['model'],
            'input': exp['input'],
            'output': exp['output'],
            'season': exp['season'],
            'metrics': {
                'rmse': float(rmse),
                'mae': float(mae),
                'r2': float(r2),
                'baseline_rmse': float(baseline_rmse),
                'skill_score': float(skill_score)
            },
            'n_samples': len(result_df),
            'training_time_sec': float(elapsed),
            'timestamp': filename.split('--')[0].replace('run_', ''),
            'input_features': input_configs[exp['input']]
        }
        
        json_filename = filename.replace('.csv', '.json')
        with open(f"{results_dir}/metadata/{json_filename}", 'w') as f:
            json.dump(metadata, f, indent=2)
        
        # Store for summary
        all_results.append({
            'timestamp': filename.split('--')[0].replace('run_', ''),
            'filename': filename,
            'model': exp['model'],
            'input': exp['input'],
            'output': exp['output'],
            'season': exp['season'] or 'all',
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'baseline_rmse': baseline_rmse,
            'skill_score': skill_score,
            'n_samples': len(result_df),
            'time_sec': elapsed
        })
        
        
    summary_df = pd.DataFrame(all_results)
summary_df.to_csv(f"{results_dir}/model_comparison_summary.csv", index=False)

# Close runlog file
runlog.close()

Progress: 465/480 (96.9%)
  ✓ Completed: 465 | ⏭️  Skipped: 0 | ✗ Failed: 0

[466/480] Lasso_alpha_0.01--ecmwf_mean_var--spi-1--Spring


Progress:  97%|█████████▋| 466/480 [59:08<00:03,  4.15it/s]

  ✓ RMSE=1.028270, MAE=0.785806, R²=-0.0248, Skill=-0.0014
  Saved: run_20260103_011950--Lasso_alpha_0_01--ecmwf_mean_var--spi-1--spring.csv

[467/480] Lasso_alpha_0.01--ecmwf_mean_var--spi-1--Summer


Progress:  98%|█████████▊| 468/480 [59:08<00:02,  4.48it/s]

  ✓ RMSE=0.879017, MAE=0.654995, R²=0.1126, Skill=0.0681
  Saved: run_20260103_011950--Lasso_alpha_0_01--ecmwf_mean_var--spi-1--summer.csv

[468/480] Lasso_alpha_0.01--ecmwf_mean_var--spi-1--Fall
  ✓ RMSE=1.049884, MAE=0.792223, R²=0.0327, Skill=0.0273
  Saved: run_20260103_011951--Lasso_alpha_0_01--ecmwf_mean_var--spi-1--fall.csv

[469/480] MLP_small_tanh--ecmwf_mean_var--spi-1--Winter


...

  ✓ RMSE=1.067856, MAE=0.855347, R²=-0.1052, Skill=-0.0400
  Saved: run_20260103_012307--MLP_2layer_small_tanh--ecmwf_mean_var--spi-1--spring.csv

[479/480] MLP_2layer_small_tanh--ecmwf_mean_var--spi-1--Summer


Progress: 100%|█████████▉| 479/480 [1:02:36<00:13, 13.28s/it]

  ✓ RMSE=0.901303, MAE=0.689979, R²=0.0670, Skill=0.0445
  Saved: run_20260103_012318--MLP_2layer_small_tanh--ecmwf_mean_var--spi-1--summer.csv

[480/480] MLP_2layer_small_tanh--ecmwf_mean_var--spi-1--Fall


Progress: 100%|██████████| 480/480 [1:02:48<00:00,  7.85s/it]

  ✓ RMSE=1.010504, MAE=0.749317, R²=0.1039, Skill=0.0638
  Saved: run_20260103_012331--MLP_2layer_small_tanh--ecmwf_mean_var--spi-1--fall.csv

✓ EXECUTION COMPLETE
Total experiments defined: 480
  ⏭️  Skipped (already run): 0
  ✓ Newly completed: 480
  ✗ Failed: 0

✓ Summary saved: 000000 Final Data/eastnor/model_results/model_comparison_summary.csv
✓ Runlog saved: 000000 Final Data/eastnor/model_results/20260103_002042_runlog.txt


# 8. Results Analysis


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os
import time
import json
from tqdm import tqdm

# Sklearn imports
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

✓ Libraries imported


## 8.7 Load All Results

In [ ]:
import glob
import os

exclude_keywords = []

results_dir = "000000 Final Data/eastnor/model_results"
metadata_files = glob.glob(f"{results_dir}/metadata/*.json")

all_results_raw = []
all_results_seasonal = []
all_results_pooled = []
skipped_count = 0

for filepath in metadata_files:
    filename = os.path.basename(filepath)
    
    # Check if filename contains any excluded keywords
    if any(keyword in filename for keyword in exclude_keywords):
        skipped_count += 1
        continue
    
    with open(filepath, 'r') as f:
        metadata = json.load(f)
    
    season = metadata.get('season') if metadata.get('season') is not None else 'all'
    result_dict = {
        'model': metadata.get('model'),
        'input': metadata.get('input'),
        'output': metadata.get('output'),
        'season': season,
        'rmse': metadata['metrics']['rmse'],
        'mae': metadata['metrics']['mae'],
        'r2': metadata['metrics']['r2'],
        'skill_score': metadata['metrics']['skill_score'],
        'n_samples': metadata.get('n_samples', 0),
    }
    all_results_raw.append(result_dict)
    # Sort results by pooled and seasonal
    if season == 'all':
        all_results_pooled.append(result_dict)
    else:
        all_results_seasonal.append(result_dict)

# Create DataFrames
results_raw_df = pd.DataFrame(all_results_raw)
results_pooled_df = pd.DataFrame(all_results_pooled)
results_seasonal_df = pd.DataFrame(all_results_seasonal)

# ------ Display: Top 20 Pooled Models ------
if not results_pooled_df.empty:
    results_pooled_sorted = results_pooled_df.sort_values('rmse').reset_index(drop=True)
    print(f"TOP 20 POOLED MODELS (by RMSE / season==all)")
    display(results_pooled_sorted[['model', 'input', 'output', 'season', 'rmse', 'mae', 'r2', 'skill_score']].head(20))

# ------ Display: Top 20 Seasonal Models ------
if not results_seasonal_df.empty:
    results_seasonal_sorted = results_seasonal_df.sort_values('rmse').reset_index(drop=True)
    print(f"TOP 20 SEASONAL MODELS (by RMSE / season!=all)")
    display(results_seasonal_sorted[['model', 'input', 'output', 'season', 'rmse', 'mae', 'r2', 'skill_score']].head(20))

# ------ Display: Worst 10 models overall ------
results_raw_df_sorted = results_raw_df.sort_values('rmse').reset_index(drop=True)
print(f"WORST 10 MODELS (by RMSE, all models)")
display(results_raw_df_sorted[['model', 'input', 'output', 'season', 'rmse', 'mae', 'r2', 'skill_score']].tail(10))

# Store in variable for further analysis
print(f"\nResults stored in variable: results_raw_df ({results_raw_df.memory_usage(deep=True).sum() / 1024:.1f} KB)")
print(f"  Columns: {list(results_raw_df.columns)}")
print(f"  You can now filter, sort, or analyze this DataFrame as needed.")


Scanning 1174 result files...
Extracting essential metrics only (efficient mode)
Including both pooled and seasonal models.

LOADED 1174 RESULTS (including 480 seasonal and 694 pooled)

Results breakdown:
  By output: {'spi-1': 587, 'wdf': 587}
  By season: {'all': 694, 'Summer': 120, 'Winter': 120, 'Fall': 120, 'Spring': 120}
  Unique models: 25
  Unique inputs: 17

TOP 20 POOLED MODELS (by RMSE / season==all)


,model,input,output,season,rmse,mae,r2,skill_score
0,Ridge_alpha_1.0,eobs,wdf,all,0.061818,0.049891,0.796802,0.550450
1,BayesianRidge,eobs,wdf,all,0.061820,0.049878,0.796790,0.550437
2,Lasso_alpha_0.0001,eobs,wdf,all,0.061820,0.049863,0.796787,0.550434
3,Ridge_alpha_0.1,eobs,wdf,all,0.061820,0.049854,0.796785,0.550431
4,LinearRegression,eobs,wdf,all,0.061821,0.049850,0.796782,0.550428
5,Lasso_alpha_0.001,eobs,wdf,all,0.061844,0.049995,0.796627,0.550256
6,Ridge_alpha_10.0,eobs,wdf,all,0.061932,0.050341,0.796048,0.549616
7,ElasticNet,eobs,wdf,all,0.062752,0.051221,0.790613,0.543656
8,Lasso_alpha_0.01,eobs,wdf,all,0.065168,0.053394,0.774181,0.526087
9,Ridge_alpha_100.0,eobs,wdf,all,0.068343,0.056695,0.751644,0.503000



TOP 20 SEASONAL MODELS (by RMSE / season!=all)


,model,input,output,season,rmse,mae,r2,skill_score
0,MLP_small_tanh,eobs,wdf,Spring,0.052830,0.041341,0.833103,0.595863
1,Ridge_alpha_1.0,eobs,wdf,Spring,0.053476,0.041872,0.828998,0.590923
2,Ridge_alpha_0.1,eobs,wdf,Spring,0.053488,0.041730,0.828922,0.590832
3,Lasso_alpha_0.0001,eobs,wdf,Spring,0.053490,0.041730,0.828904,0.590811
4,LinearRegression,eobs,wdf,Spring,0.053491,0.041715,0.828904,0.590810
5,Lasso_alpha_0.001,eobs,wdf,Spring,0.053518,0.041874,0.828726,0.590597
6,Ridge_alpha_10.0,eobs,wdf,Spring,0.054412,0.043389,0.822956,0.583759
7,Lasso_alpha_0.01,eobs,wdf,Spring,0.055113,0.043507,0.818370,0.578401
8,Lasso_alpha_0.001,eobs,wdf,Winter,0.056452,0.045688,0.765950,0.521530
9,Lasso_alpha_0.0001,eobs,wdf,Winter,0.056588,0.045720,0.764819,0.520374



WORST 10 MODELS (by RMSE, all models)


,model,input,output,season,rmse,mae,r2,skill_score
1164,Lasso_alpha_0.0001,eobs_lag1,spi-1,Fall,1.102710,0.849678,-0.067137,-0.021671
1165,Lasso_alpha_0.001,eobs_lag1,spi-1,Fall,1.102747,0.849577,-0.067208,-0.021705
1166,MLP_small_tanh,ecmwf_mean_var,spi-1,Fall,1.103281,0.825240,-0.068242,-0.022200
1167,MLP_2layer_small_tanh,ecmwf_mean_eobs_lag2,spi-1,Spring,1.107460,0.872068,-0.188681,-0.078543
1168,MLP_2layer_small,all_features,spi-1,all,1.110066,0.893168,-0.268633,-0.123276
1169,MLP_small_tanh,ecmwf_mean_eobs_lag2,spi-1,Spring,1.113456,0.848106,-0.201587,-0.084383
1170,MLP_small_tanh,eobs_lag1,spi-1,Fall,1.114011,0.855758,-0.089122,-0.032142
1171,MLP_small_tanh,ecmwf_mean_eobs_lag1,spi-1,Fall,1.116960,0.865606,-0.094894,-0.034873
1172,MLP_medium_tanh,ecmwf_mean_eobs_lag1,spi-1,Fall,1.118171,0.851654,-0.097270,-0.035995
1173,MLP_small_tanh,ecmwf_mean_eobs_lag1,spi-1,Spring,1.130070,0.874738,-0.237713,-0.100563



✓ Results stored in variable: results_raw_df (351.7 KB)
  Columns: ['model', 'input', 'output', 'season', 'rmse', 'mae', 'r2', 'skill_score', 'n_samples']
  You can now filter, sort, or analyze this DataFrame as needed.
  (See also: results_pooled_df, results_seasonal_df)


### WDF

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

wdf_df = sorted_df[sorted_df['output'] == 'wdf'].reset_index(drop=True)

print(wdf_df.to_string(index=False, line_width=sys.maxsize))

pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

                model                         input output season     rmse      mae        r2  skill_score  n_samples
       MLP_small_tanh                          eobs    wdf Spring 0.052830 0.041341  0.833103     0.595863         93
      Ridge_alpha_1.0                          eobs    wdf Spring 0.053476 0.041872  0.828998     0.590923         93
      Ridge_alpha_0.1                          eobs    wdf Spring 0.053488 0.041730  0.828922     0.590832         93
   Lasso_alpha_0.0001                          eobs    wdf Spring 0.053490 0.041730  0.828904     0.590811         93
     LinearRegression                          eobs    wdf Spring 0.053491 0.041715  0.828904     0.590810         93
    Lasso_alpha_0.001                          eobs    wdf Spring 0.053518 0.041874  0.828726     0.590597         93
     Ridge_alpha_10.0                          eobs    wdf Spring 0.054412 0.043389  0.822956     0.583759         93
     Lasso_alpha_0.01                          eobs    w

### SPI-1

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

wdf_df = sorted_df[sorted_df['output'] == 'spi-1'].reset_index(drop=True)

print(wdf_df.to_string(index=False, line_width=sys.maxsize))

pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

                model                         input output season     rmse      mae        r2  skill_score  n_samples
       MLP_small_tanh                          eobs  spi-1   Fall 0.294959 0.173188  0.923648     0.726718         91
MLP_2layer_small_tanh                          eobs  spi-1   Fall 0.315441 0.186338  0.912676     0.707741         91
       MLP_small_tanh                          eobs  spi-1 Summer 0.282355 0.219845  0.908438     0.700662         93
      Ridge_alpha_1.0                          eobs  spi-1   Fall 0.332586 0.228573  0.902925     0.691856         91
     Lasso_alpha_0.01                          eobs  spi-1   Fall 0.332725 0.228534  0.902844     0.691727         91
    Lasso_alpha_0.001                          eobs  spi-1   Fall 0.332815 0.227308  0.902792     0.691644         91
      Ridge_alpha_0.1                          eobs  spi-1   Fall 0.332816 0.227313  0.902791     0.691643         91
   Lasso_alpha_0.0001                          eobs  spi